# Stress prediction from physiological signals (extension)

Here the goal is to use metadata about the subjects, as well as their survey data as a unified static personal traits. This would adress the problem of different type of stresses and reaction based on the person, ideally aiming to create personalised models.

### 0. Imports

In [1]:
import os
import pandas as pd

### 1. Metadata from readme.txt

In [2]:
DATA_ROOT = "./WESAD"

metadata_txt = {"Subject": [], "Text": []}
metadata_files = sorted([
    os.path.join(root, f)
    for root, dirs, files in os.walk(DATA_ROOT)
    for f in files if f.endswith("readme.txt") and f.startswith("S")
])

print(f"Found {len(metadata_files)} metadata files:")

subjects = [os.path.basename(os.path.dirname(f)) for f in metadata_files]
print(f"Subjects: {subjects}")

for f in metadata_files:
    with open(f, "r") as file:
        content = file.read()
    metadata_txt["Subject"].append(os.path.basename(os.path.dirname(f)))
    metadata_txt["Text"].append(content)

print(f"Metadata entries: {len(metadata_txt['Subject'])}")

print("\nSample metadata entry:")
print("\n'''")
print(f"Subject: {metadata_txt['Subject'][0]}")
print(f"Text: {metadata_txt['Text'][0]}")
print("'''")

Found 15 metadata files:
Subjects: ['S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']
Metadata entries: 15

Sample metadata entry:

'''
Subject: S10
Text: ### Personal information ###
Age: 28
Height (cm): 178
Weight (kg): 76
Gender: male
Dominant hand: right

### Study pre-requisites ###
Did you drink coffee today? NO
Did you drink coffee within the last hour? NO
Did you do any sports today? NO
Are you a smoker? NO
Did you smoke within the last hour? NO
Do you feel ill today? NO

### Additional notes ###
-

'''


In [3]:
# we want to extract the following information from the metadata:

metadata = pd.DataFrame()
text_series = pd.Series(metadata_txt["Text"])

# - Subject ID
metadata["subject_id"] = metadata_txt["Subject"]
# - Age
extract_age = lambda text: int(next((line.split(":")[1].strip() for line in text.splitlines() if "Age" in line), None))
metadata["age"] = text_series.apply(lambda x: extract_age(x))
# - Height
extract_height = lambda text: int(next((line.split(":")[1].strip() for line in text.splitlines() if "Height" in line), None))
metadata["height"] = text_series.apply(lambda x: extract_height(x))
# - Weight
extract_weight = lambda text: int(next((line.split(":")[1].strip() for line in text.splitlines() if "Weight" in line), None))
metadata["weight"] = text_series.apply(lambda x: extract_weight(x))
# - BMI
metadata["bmi"] = round(metadata["weight"] / (metadata["height"] / 100) ** 2, 1)
# - Gender
extract_gender = lambda text: next((line.split(":")[1].strip() for line in text.splitlines() if "Gender" in line), None)
metadata["gender"] = text_series.apply(lambda x: extract_gender(x))
# - Coffee today?
extract_coffee = lambda text: next((line.split("?")[1].strip() for line in text.splitlines() if "coffee today" in line), None)
metadata["coffee_today"] = text_series.apply(lambda x: extract_coffee(x))
# - Coffee last hour?
extract_coffee_last_hour = lambda text: next((line.split("?")[1].strip() for line in text.splitlines() if "coffee within the last hour" in line), None)
metadata["coffee_last_hour"] = text_series.apply(lambda x: extract_coffee_last_hour(x))
# - Sports today?
extract_sports = lambda text: next((line.split("?")[1].strip() for line in text.splitlines() if "sports today" in line), None)
metadata["sports_today"] = text_series.apply(lambda x: extract_sports(x))
# - Smoker?
extract_smoker = lambda text: next((line.split("?")[1].strip() for line in text.splitlines() if "smoker" in line), None)
metadata["smoker"] = text_series.apply(lambda x: extract_smoker(x))
# - Smoke last hour?
extract_smoke_last_hour = lambda text: next((line.split("?")[1].strip() for line in text.splitlines() if "smoke within the last hour" in line), None)
metadata["smoke_last_hour"] = text_series.apply(lambda x: extract_smoke_last_hour(x))
# - Ill today?
extract_ill = lambda text: next((line.split("?")[1].strip() for line in text.splitlines() if "ill today" in line), None)
metadata["ill_today"] = text_series.apply(lambda x: extract_ill(x))

print("\nExtracted metadata:")
display(metadata)



Extracted metadata:


,subject_id,age,height,weight,bmi,gender,coffee_today,coffee_last_hour,sports_today,smoker,smoke_last_hour,ill_today
0,S10,28,178,76,24.0,male,NO,NO,NO,NO,NO,NO
1,S11,26,171,54,18.5,female,YES,NO,NO,NO,NO,NO
2,S13,28,181,82,25.0,male,NO,NO,NO,NO,NO,NO
3,S14,27,180,80,24.7,male,NO,NO,NO,NO,NO,NO
4,S15,28,186,83,24.0,male,NO,NO,NO,NO,NO,NO
5,S16,24,184,69,20.4,male,NO,NO,NO,NO,NO,NO
6,S17,29,165,55,20.2,female,NO,NO,NO,NO,NO,NO
7,S2,27,175,80,26.1,male,NO,NO,NO,NO,NO,NO
8,S3,27,173,69,23.1,male,NO,NO,NO,NO,NO,NO
9,S4,25,175,90,29.4,male,NO,NO,NO,NO,NO,NO


In [4]:
# one hot encode the categorical variables except subject_id
metadata_encoded = pd.get_dummies(metadata.drop(columns=["subject_id"]), drop_first=True, dtype=int)
metadata_encoded["subject_id"] = metadata["subject_id"]
cols = ["subject_id"] + [col for col in metadata_encoded.columns if col != "subject_id"]
metadata_encoded = metadata_encoded[cols]

print("\nOne-hot encoded metadata (Numeric):")
display(metadata_encoded)


One-hot encoded metadata (Numeric):


,subject_id,age,height,weight,bmi,gender_male,coffee_today_YES,sports_today_YES,smoker_YES,ill_today_YES
0,S10,28,178,76,24.0,1,0,0,0,0
1,S11,26,171,54,18.5,0,1,0,0,0
2,S13,28,181,82,25.0,1,0,0,0,0
3,S14,27,180,80,24.7,1,0,0,0,0
4,S15,28,186,83,24.0,1,0,0,0,0
5,S16,24,184,69,20.4,1,0,0,0,0
6,S17,29,165,55,20.2,0,0,0,0,0
7,S2,27,175,80,26.1,1,0,0,0,0
8,S3,27,173,69,23.1,1,0,0,0,0
9,S4,25,175,90,29.4,1,0,0,0,0


### 2. Survey data

To integrate the survey data into your subject metadata cleanly, we will treat the survey scores as static psychological traits. We will pull specific values (like the Baseline scores) and aggregate others (like the range of emotions) to create a single row for each subject.

In [5]:
# show all columns in pandas
pd.set_option('display.max_columns', None)

subject = "S2"

# example of csv
path = f"WESAD/{subject}/{subject}_quest.csv"
example_survey = pd.read_csv(path, sep=';', header=None)
print(f"\nExample survey data from {path}:")
display(example_survey)


Example survey data from WESAD/S2/S2_quest.csv:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26
0,# Subj,S2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,# ORDER,Base,TSST,Medi 1,Fun,Medi 2,sRead,fRead,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,# START,7.08,39.55,70.19,81.25,93.38,54.42,89.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,# END,26.32,50.3,77.1,87.47,100.15,56.07,91.15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,# PANAS,1,1,3,2,1,3,1,1.0,1.0,2.0,2.0,2.0,2.0,1.0,4.0,3.0,4.0,4.0,2.0,2.0,2.0,1.0,2.0,1.0,NaN,NaN
6,# PANAS,3,2,4,1,3,3,1,2.0,1.0,4.0,2.0,4.0,3.0,1.0,5.0,4.0,4.0,4.0,2.0,3.0,3.0,3.0,2.0,1.0,3.0,1.0
7,# PANAS,1,1,2,3,1,2,1,1.0,1.0,1.0,1.0,1.0,3.0,1.0,2.0,1.0,2.0,3.0,1.0,1.0,1.0,1.0,4.0,1.0,NaN,NaN
8,# PANAS,1,1,2,3,1,1,1,1.0,1.0,1.0,1.0,1.0,2.0,1.0,4.0,1.0,1.0,3.0,1.0,1.0,1.0,2.0,3.0,1.0,NaN,NaN
9,# PANAS,1,1,1,2,1,1,1,1.0,1.0,1.0,1.0,1.0,2.0,1.0,2.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,3.0,1.0,NaN,NaN


In [6]:
# for each lable (Base, Stress, Amusement, Meditation) we want to extract:
# 5 PANAS
# 5 STAI
# 1 SSSQ
# DIM will only be for Base and Stress or Amusement depending on fun_first, rest is 0.

# first of all let's remove row 0 then set current row 0 to be the column names
example_survey = example_survey.drop(index=0).reset_index(drop=True)
example_survey.columns = example_survey.iloc[0]
example_survey = example_survey.drop(index=0).reset_index(drop=True)

# also remove first column
example_survey = example_survey.drop(columns=example_survey.columns[0])

# add a column to metadata that says "fun_first" 0 or 1, because subjects had either "Fun" or "TSST" as their first affective state
# can be seen based on what is the first column name after base rn
# if first column after "Base" is "Fun", then fun_first is 1, else 0
fun_first = 1 if example_survey.columns[1] == "Fun" else 0 if example_survey.columns[1] == "TSST" else None
print(f"\nFun first: {fun_first}")

# only keep "Base", "TSST", "Fun", "Medi 1", "Medi 2" columns
example_survey = example_survey[["Base", "TSST", "Fun", "Medi 1", "Medi 2"]]

# remove time rows (row 0 and row 1) and rows where all are NaN
example_survey = example_survey.drop(index=[0, 1]).reset_index(drop=True)
example_survey = example_survey.dropna(how='all').reset_index(drop=True)

# replace NaN with 0
example_survey = example_survey.fillna(0)
display(example_survey)


Fun first: 0


,Base,TSST,Fun,Medi 1,Medi 2
0,1,1,2,3,1
1,3,2,1,4,3
2,1,1,3,2,1
3,1,1,3,2,1
4,1,1,2,1,1
5,3,2,4,1,2
6,1,3,1,2,3
7,4,1,4,1,2
8,3,1,3,1,1
9,4,1,4,1,1


Since we have fun_first now, we can make sure its always in same order without losing that information.\
Let's always make it Base - TSST - Fun - Medi 1 - Medi 2

In [7]:
# then we want to turn into 1d
transposed = example_survey.T
flattened_ordered = transposed.stack()
flattened_ordered.index = [f"{cond}_{idx}" for cond in example_survey.columns for idx in example_survey.index]
flattened_ordered = flattened_ordered.to_frame().T
print(flattened_ordered.shape)
display(flattened_ordered)

(1, 80)


,Base_0,Base_1,Base_2,Base_3,Base_4,Base_5,Base_6,Base_7,Base_8,Base_9,Base_10,Base_11,Base_12,Base_13,Base_14,Base_15,TSST_0,TSST_1,TSST_2,TSST_3,TSST_4,TSST_5,TSST_6,TSST_7,TSST_8,TSST_9,TSST_10,TSST_11,TSST_12,TSST_13,TSST_14,TSST_15,Fun_0,Fun_1,Fun_2,Fun_3,Fun_4,Fun_5,Fun_6,Fun_7,Fun_8,Fun_9,Fun_10,Fun_11,Fun_12,Fun_13,Fun_14,Fun_15,Medi 1_0,Medi 1_1,Medi 1_2,Medi 1_3,Medi 1_4,Medi 1_5,Medi 1_6,Medi 1_7,Medi 1_8,Medi 1_9,Medi 1_10,Medi 1_11,Medi 1_12,Medi 1_13,Medi 1_14,Medi 1_15,Medi 2_0,Medi 2_1,Medi 2_2,Medi 2_3,Medi 2_4,Medi 2_5,Medi 2_6,Medi 2_7,Medi 2_8,Medi 2_9,Medi 2_10,Medi 2_11,Medi 2_12,Medi 2_13,Medi 2_14,Medi 2_15
0,1,3,1,1,1,3,1,4,3,4,7,5,7,8,7,5,1,2,1,1,1,2,3,1,1,1,2,4,2,1,2,5,2,1,3,3,2,4,1,4,3,4,0,0,0,0,0,4,3,4,2,2,1,1,2,1,1,1,0,0,0,0,0,4,1,3,1,1,1,2,3,2,1,1,0,0,0,0,0,3


In [8]:
# then we can add as column 81 fun_first
flattened_ordered["fun_first"] = fun_first

# and column 82 subject_id
flattened_ordered["subject_id"] = subject
print(flattened_ordered.shape)
display(flattened_ordered)

(1, 82)


,Base_0,Base_1,Base_2,Base_3,Base_4,Base_5,Base_6,Base_7,Base_8,Base_9,Base_10,Base_11,Base_12,Base_13,Base_14,Base_15,TSST_0,TSST_1,TSST_2,TSST_3,TSST_4,TSST_5,TSST_6,TSST_7,TSST_8,TSST_9,TSST_10,TSST_11,TSST_12,TSST_13,TSST_14,TSST_15,Fun_0,Fun_1,Fun_2,Fun_3,Fun_4,Fun_5,Fun_6,Fun_7,Fun_8,Fun_9,Fun_10,Fun_11,Fun_12,Fun_13,Fun_14,Fun_15,Medi 1_0,Medi 1_1,Medi 1_2,Medi 1_3,Medi 1_4,Medi 1_5,Medi 1_6,Medi 1_7,Medi 1_8,Medi 1_9,Medi 1_10,Medi 1_11,Medi 1_12,Medi 1_13,Medi 1_14,Medi 1_15,Medi 2_0,Medi 2_1,Medi 2_2,Medi 2_3,Medi 2_4,Medi 2_5,Medi 2_6,Medi 2_7,Medi 2_8,Medi 2_9,Medi 2_10,Medi 2_11,Medi 2_12,Medi 2_13,Medi 2_14,Medi 2_15,fun_first,subject_id
0,1,3,1,1,1,3,1,4,3,4,7,5,7,8,7,5,1,2,1,1,1,2,3,1,1,1,2,4,2,1,2,5,2,1,3,3,2,4,1,4,3,4,0,0,0,0,0,4,3,4,2,2,1,1,2,1,1,1,0,0,0,0,0,4,1,3,1,1,1,2,3,2,1,1,0,0,0,0,0,3,0,S2


In [9]:
survey_df = pd.DataFrame()
for subject in subjects:
    path = f"WESAD/{subject}/{subject}_quest.csv"
    # read
    survey = pd.read_csv(path, sep=';', header=None)
    # drop row 0 then set current row 0 to be the column names
    survey = survey.drop(index=0).reset_index(drop=True)
    survey.columns = survey.iloc[0]
    survey = survey.drop(index=0).reset_index(drop=True)
    survey = survey.drop(columns=survey.columns[0])
    # determine if fun_first is 1 or 0 based on what is the first column name after "Base"
    fun_first = 1 if survey.columns[1] == "Fun" else 0 if survey.columns[1] == "TSST" else None
    survey = survey[["Base", "TSST", "Fun", "Medi 1", "Medi 2"]]
    survey = survey.drop(index=[0, 1]).reset_index(drop=True)
    survey = survey.dropna(how='all').reset_index(drop=True)
    survey = survey.fillna(0)
    # turn to 1d
    transposed = survey.T
    flattened_ordered = transposed.stack()
    flattened_ordered.index = [f"{cond}_{idx}" for cond in survey.columns for idx in survey.index]
    flattened_ordered = flattened_ordered.to_frame().T
    # add fun_first column
    flattened_ordered["fun_first"] = fun_first
    # add subject_id column
    flattened_ordered["subject_id"] = subject
    # append to survey_df
    survey_df = pd.concat([survey_df, flattened_ordered], ignore_index=True)


print(survey_df.shape)
display(survey_df)

(15, 82)


,Base_0,Base_1,Base_2,Base_3,Base_4,Base_5,Base_6,Base_7,Base_8,Base_9,Base_10,Base_11,Base_12,Base_13,Base_14,Base_15,TSST_0,TSST_1,TSST_2,TSST_3,TSST_4,TSST_5,TSST_6,TSST_7,TSST_8,TSST_9,TSST_10,TSST_11,TSST_12,TSST_13,TSST_14,TSST_15,Fun_0,Fun_1,Fun_2,Fun_3,Fun_4,Fun_5,Fun_6,Fun_7,Fun_8,Fun_9,Fun_10,Fun_11,Fun_12,Fun_13,Fun_14,Fun_15,Medi 1_0,Medi 1_1,Medi 1_2,Medi 1_3,Medi 1_4,Medi 1_5,Medi 1_6,Medi 1_7,Medi 1_8,Medi 1_9,Medi 1_10,Medi 1_11,Medi 1_12,Medi 1_13,Medi 1_14,Medi 1_15,Medi 2_0,Medi 2_1,Medi 2_2,Medi 2_3,Medi 2_4,Medi 2_5,Medi 2_6,Medi 2_7,Medi 2_8,Medi 2_9,Medi 2_10,Medi 2_11,Medi 2_12,Medi 2_13,Medi 2_14,Medi 2_15,fun_first,subject_id
0,2,2,1,4,2,3,3,4,1,3,6,8,6,3,6,3,1,4,2,1,1,3,3,3,1,3,0,0,0,0,0,3,1,1,1,4,2,2,1,1,4,2,2,2,1,8,2,2,3,3,2,3,1,1,1,1,2,1,0,0,0,0,0,2,1,1,1,3,1,1,1,1,2,1,0,0,0,0,0,3,1,S10
1,1,4,2,1,2,3,1,4,4,4,6,4,7,8,6,3,1,1,1,1,1,1,3,1,1,1,2,6,2,3,2,5,2,2,3,4,3,4,1,4,4,4,0,0,0,0,0,3,3,4,4,3,3,1,2,1,1,1,0,0,0,0,0,4,1,1,2,1,1,1,2,1,1,1,0,0,0,0,0,4,0,S11
2,1,1,1,5,1,2,4,4,1,2,5,8,8,4,2,4,1,4,2,1,1,3,3,4,1,2,0,0,0,0,0,4,1,1,1,2,2,2,1,1,4,2,3,4,1,9,6,5,4,3,2,4,1,2,2,1,3,1,0,0,0,0,0,5,1,1,1,2,2,1,1,1,2,3,0,0,0,0,0,3,1,S13
3,1,4,1,1,1,1,1,1,1,1,5,3,4,7,7,5,1,1,1,1,1,1,3,2,1,1,2,7,3,2,2,5,1,1,1,2,1,3,1,4,3,3,0,0,0,0,0,2,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,3,3,3,3,3,3,2,2,2,2,1,0,0,0,0,0,5,0,S14
4,2,2,1,4,2,2,3,4,1,4,6,7,8,4,7,4,2,3,3,1,3,3,3,4,1,4,0,0,0,0,0,4,1,1,1,3,1,2,2,1,3,1,5,5,2,8,4,4,3,3,2,2,2,1,1,2,2,1,0,0,0,0,0,4,1,1,1,2,1,1,1,1,2,1,0,0,0,0,0,4,1,S15
5,2,4,2,1,1,2,2,3,2,2,7,3,8,7,7,3,1,3,1,2,1,2,4,1,1,1,2,8,2,4,1,3,2,2,2,3,2,3,1,4,3,4,0,0,0,0,0,4,3,3,2,3,2,2,3,1,2,1,0,0,0,0,0,4,1,2,1,1,1,1,2,2,1,1,0,0,0,0,0,3,0,S16
6,2,3,1,5,4,2,3,3,1,2,7,8,7,1,5,3,1,4,1,1,1,2,4,3,1,2,0,0,0,0,0,4,1,1,1,2,1,1,1,1,4,2,2,2,2,9,2,4,4,4,3,1,2,2,1,1,2,1,0,0,0,0,0,1,1,1,1,5,1,1,1,1,4,1,0,0,0,0,0,5,1,S17
7,1,3,1,1,1,3,1,4,3,4,7,5,7,8,7,5,1,2,1,1,1,2,3,1,1,1,2,4,2,1,2,5,2,1,3,3,2,4,1,4,3,4,0,0,0,0,0,4,3,4,2,2,1,1,2,1,1,1,0,0,0,0,0,4,1,3,1,1,1,2,3,2,1,1,0,0,0,0,0,3,0,S2
8,3,4,1,4,3,3,2,3,4,4,7,7,6,8,6,4,1,4,3,1,1,1,4,1,1,1,4,7,3,4,3,5,3,3,2,5,1,4,1,2,3,3,0,0,0,0,0,4,5,5,2,5,2,1,3,1,3,3,0,0,0,0,0,4,1,1,1,1,1,2,2,1,1,1,0,0,0,0,0,5,0,S3
9,1,1,1,4,1,3,3,1,1,2,7,8,5,5,5,3,2,4,1,1,2,4,4,2,2,3,0,0,0,0,0,4,2,1,1,2,2,1,1,1,3,1,2,1,7,7,2,4,3,4,1,2,1,1,1,1,2,1,0,0,0,0,0,3,2,1,2,1,2,1,1,2,1,2,0,0,0,0,0,5,1,S4


### 3. Merge both

In [10]:
# merge survey_df with metadata_encoded on subject_id
final_df = pd.merge(survey_df, metadata_encoded, on="subject_id", how="left")
# make sure subject_id is the first column
cols = ["subject_id"] + [col for col in final_df.columns if col != "subject_id"]
final_df = final_df[cols]
print(survey_df.shape, "+", metadata_encoded.shape, "- the double counting of subject_id")
print(final_df.shape)
display(final_df)

(15, 82) + (15, 10) - the double counting of subject_id
(15, 91)


,subject_id,Base_0,Base_1,Base_2,Base_3,Base_4,Base_5,Base_6,Base_7,Base_8,Base_9,Base_10,Base_11,Base_12,Base_13,Base_14,Base_15,TSST_0,TSST_1,TSST_2,TSST_3,TSST_4,TSST_5,TSST_6,TSST_7,TSST_8,TSST_9,TSST_10,TSST_11,TSST_12,TSST_13,TSST_14,TSST_15,Fun_0,Fun_1,Fun_2,Fun_3,Fun_4,Fun_5,Fun_6,Fun_7,Fun_8,Fun_9,Fun_10,Fun_11,Fun_12,Fun_13,Fun_14,Fun_15,Medi 1_0,Medi 1_1,Medi 1_2,Medi 1_3,Medi 1_4,Medi 1_5,Medi 1_6,Medi 1_7,Medi 1_8,Medi 1_9,Medi 1_10,Medi 1_11,Medi 1_12,Medi 1_13,Medi 1_14,Medi 1_15,Medi 2_0,Medi 2_1,Medi 2_2,Medi 2_3,Medi 2_4,Medi 2_5,Medi 2_6,Medi 2_7,Medi 2_8,Medi 2_9,Medi 2_10,Medi 2_11,Medi 2_12,Medi 2_13,Medi 2_14,Medi 2_15,fun_first,age,height,weight,bmi,gender_male,coffee_today_YES,sports_today_YES,smoker_YES,ill_today_YES
0,S10,2,2,1,4,2,3,3,4,1,3,6,8,6,3,6,3,1,4,2,1,1,3,3,3,1,3,0,0,0,0,0,3,1,1,1,4,2,2,1,1,4,2,2,2,1,8,2,2,3,3,2,3,1,1,1,1,2,1,0,0,0,0,0,2,1,1,1,3,1,1,1,1,2,1,0,0,0,0,0,3,1,28,178,76,24.0,1,0,0,0,0
1,S11,1,4,2,1,2,3,1,4,4,4,6,4,7,8,6,3,1,1,1,1,1,1,3,1,1,1,2,6,2,3,2,5,2,2,3,4,3,4,1,4,4,4,0,0,0,0,0,3,3,4,4,3,3,1,2,1,1,1,0,0,0,0,0,4,1,1,2,1,1,1,2,1,1,1,0,0,0,0,0,4,0,26,171,54,18.5,0,1,0,0,0
2,S13,1,1,1,5,1,2,4,4,1,2,5,8,8,4,2,4,1,4,2,1,1,3,3,4,1,2,0,0,0,0,0,4,1,1,1,2,2,2,1,1,4,2,3,4,1,9,6,5,4,3,2,4,1,2,2,1,3,1,0,0,0,0,0,5,1,1,1,2,2,1,1,1,2,3,0,0,0,0,0,3,1,28,181,82,25.0,1,0,0,0,0
3,S14,1,4,1,1,1,1,1,1,1,1,5,3,4,7,7,5,1,1,1,1,1,1,3,2,1,1,2,7,3,2,2,5,1,1,1,2,1,3,1,4,3,3,0,0,0,0,0,2,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0,3,3,3,3,3,3,2,2,2,2,1,0,0,0,0,0,5,0,27,180,80,24.7,1,0,0,0,0
4,S15,2,2,1,4,2,2,3,4,1,4,6,7,8,4,7,4,2,3,3,1,3,3,3,4,1,4,0,0,0,0,0,4,1,1,1,3,1,2,2,1,3,1,5,5,2,8,4,4,3,3,2,2,2,1,1,2,2,1,0,0,0,0,0,4,1,1,1,2,1,1,1,1,2,1,0,0,0,0,0,4,1,28,186,83,24.0,1,0,0,0,0
5,S16,2,4,2,1,1,2,2,3,2,2,7,3,8,7,7,3,1,3,1,2,1,2,4,1,1,1,2,8,2,4,1,3,2,2,2,3,2,3,1,4,3,4,0,0,0,0,0,4,3,3,2,3,2,2,3,1,2,1,0,0,0,0,0,4,1,2,1,1,1,1,2,2,1,1,0,0,0,0,0,3,0,24,184,69,20.4,1,0,0,0,0
6,S17,2,3,1,5,4,2,3,3,1,2,7,8,7,1,5,3,1,4,1,1,1,2,4,3,1,2,0,0,0,0,0,4,1,1,1,2,1,1,1,1,4,2,2,2,2,9,2,4,4,4,3,1,2,2,1,1,2,1,0,0,0,0,0,1,1,1,1,5,1,1,1,1,4,1,0,0,0,0,0,5,1,29,165,55,20.2,0,0,0,0,0
7,S2,1,3,1,1,1,3,1,4,3,4,7,5,7,8,7,5,1,2,1,1,1,2,3,1,1,1,2,4,2,1,2,5,2,1,3,3,2,4,1,4,3,4,0,0,0,0,0,4,3,4,2,2,1,1,2,1,1,1,0,0,0,0,0,4,1,3,1,1,1,2,3,2,1,1,0,0,0,0,0,3,0,27,175,80,26.1,1,0,0,0,0
8,S3,3,4,1,4,3,3,2,3,4,4,7,7,6,8,6,4,1,4,3,1,1,1,4,1,1,1,4,7,3,4,3,5,3,3,2,5,1,4,1,2,3,3,0,0,0,0,0,4,5,5,2,5,2,1,3,1,3,3,0,0,0,0,0,4,1,1,1,1,1,2,2,1,1,1,0,0,0,0,0,5,0,27,173,69,23.1,1,0,0,0,0
9,S4,1,1,1,4,1,3,3,1,1,2,7,8,5,5,5,3,2,4,1,1,2,4,4,2,2,3,0,0,0,0,0,4,2,1,1,2,2,1,1,1,3,1,2,1,7,7,2,4,3,4,1,2,1,1,1,1,2,1,0,0,0,0,0,3,2,1,2,1,2,1,1,2,1,2,0,0,0,0,0,5,1,25,175,90,29.4,1,0,0,0,0


Great, now let's merge with signals

In [11]:
# take wesad_wrist_features.csv
features_path = "wesad_wrist_features.csv"
signals = pd.read_csv(features_path)
print(f"\nWrist features data from {features_path}:")
print(signals.shape)
display(signals.head())


Wrist features data from wesad_wrist_features.csv:
(179817, 46)


,subject,window_index,label,bvp_hr_mean,bvp_hr_std,bvp_hrv_mean,bvp_hrv_std,bvp_hrv_nn50,bvp_hrv_pnn50,bvp_hrv_rmssd,bvp_hrv_ulf,bvp_hrv_lf,bvp_hrv_hf,bvp_hrv_uhf,bvp_hrv_lf_hf,eda_mean,eda_std,eda_min,eda_max,eda_slope,eda_range,eda_scr_peaks,eda_scr_mean_amp,eda_scr_auc,temp_mean,temp_std,temp_min,temp_max,temp_range,temp_slope,acc_x_mean,acc_x_std,acc_y_mean,acc_y_std,acc_z_mean,acc_z_std,acc_mag_mean,acc_mag_std,acc_x_absint,acc_y_absint,acc_z_absint,acc_x_peakfreq,acc_y_peakfreq,acc_z_peakfreq,eda_scr_has_peaks,bvp_hrv_freq_valid
0,S10,0,1,92.222633,30.733178,729.552469,259.659106,60,0.750000,391.919729,0.001551,0.005276,0.025225,0.009141,0.209149,0.388696,0.044754,0.34759,0.606228,0.001623,0.258638,7,0.062677,0.415006,33.247683,0.048199,33.176,33.322,0.146,0.002654,39.746387,12.666104,2.475993,3.129106,41.014355,19.997278,61.879907,3.012126,2384.783203,202.807617,2570.923828,0.125,0.125,0.125,1,1
1,S10,1,1,92.222633,30.733178,729.552469,259.659106,60,0.750000,391.919729,0.001551,0.005276,0.025225,0.009141,0.209149,0.388972,0.044758,0.34759,0.606228,0.001624,0.258638,7,0.062677,0.414869,33.248181,0.048316,33.176,33.322,0.146,0.002663,39.630241,12.626917,2.457064,3.110095,41.140055,19.991351,61.877399,3.008428,2377.814453,201.671875,2578.465820,0.125,0.125,0.125,1,1
2,S10,2,1,92.743837,30.566826,723.828125,256.147262,59,0.746835,389.281709,0.001605,0.005238,0.025854,0.009290,0.202613,0.389237,0.044763,0.34759,0.606228,0.001624,0.258638,7,0.062677,0.414794,33.248677,0.048440,33.176,33.322,0.146,0.002673,39.513574,12.586034,2.439437,3.092266,41.265267,19.984413,61.873905,3.010429,2370.814453,200.614258,2585.978516,0.125,0.125,0.125,1,1
3,S10,3,1,92.652350,30.388575,723.572531,254.571460,59,0.737500,386.844992,0.001604,0.005119,0.025099,0.009718,0.203935,0.389497,0.044764,0.34759,0.606228,0.001625,0.258638,7,0.062677,0.414637,33.249171,0.048570,33.176,33.322,0.146,0.002684,39.397217,12.544515,2.423535,3.075650,41.390365,19.976378,61.870738,3.011397,2363.833008,199.660156,2593.484375,0.125,0.125,0.125,1,1
4,S10,4,1,92.629447,30.394781,723.765432,254.561813,59,0.737500,386.856825,0.001603,0.005119,0.025099,0.009718,0.203940,0.389762,0.044763,0.34759,0.606228,0.001625,0.258638,7,0.062677,0.414685,33.249662,0.048708,33.176,33.322,0.146,0.002695,39.281429,12.502663,2.409635,3.060331,41.516048,19.967527,61.868780,3.009604,2356.885742,198.826172,2601.025391,0.125,0.125,0.125,1,1


In [12]:
# one version with only metadata added
# first we rename subject_id to subject in metadata_encoded to match signals
metadata_encoded.rename(columns={"subject_id": "subject"}, inplace=True)
merged_metadata_df = pd.merge(metadata_encoded, signals, on="subject", how="left")
print(signals.shape, "+", metadata_encoded.shape, "- the double counting of subject_id")
print(merged_metadata_df.shape)
display(merged_metadata_df)

# save merged_metadata_df to csv
merged_metadata_df.to_csv("merged_metadata_wrist_features.csv", index=False)

(179817, 46) + (15, 10) - the double counting of subject_id
(179817, 55)


,subject,age,height,weight,bmi,gender_male,coffee_today_YES,sports_today_YES,smoker_YES,ill_today_YES,window_index,label,bvp_hr_mean,bvp_hr_std,bvp_hrv_mean,bvp_hrv_std,bvp_hrv_nn50,bvp_hrv_pnn50,bvp_hrv_rmssd,bvp_hrv_ulf,bvp_hrv_lf,bvp_hrv_hf,bvp_hrv_uhf,bvp_hrv_lf_hf,eda_mean,eda_std,eda_min,eda_max,eda_slope,eda_range,eda_scr_peaks,eda_scr_mean_amp,eda_scr_auc,temp_mean,temp_std,temp_min,temp_max,temp_range,temp_slope,acc_x_mean,acc_x_std,acc_y_mean,acc_y_std,acc_z_mean,acc_z_std,acc_mag_mean,acc_mag_std,acc_x_absint,acc_y_absint,acc_z_absint,acc_x_peakfreq,acc_y_peakfreq,acc_z_peakfreq,eda_scr_has_peaks,bvp_hrv_freq_valid
0,S10,28,178,76,24.0,1,0,0,0,0,0,1,92.222633,30.733178,729.552469,259.659106,60,0.750000,391.919729,0.001551,0.005276,0.025225,0.009141,0.209149,0.388696,0.044754,0.347590,0.606228,0.001623,0.258638,7,0.062677,0.415006,33.247683,0.048199,33.176,33.322,0.146,0.002654,39.746387,12.666104,2.475993,3.129106,41.014355,19.997278,61.879907,3.012126,2384.783203,202.807617,2570.923828,0.125,0.125,0.125,1,1
1,S10,28,178,76,24.0,1,0,0,0,0,1,1,92.222633,30.733178,729.552469,259.659106,60,0.750000,391.919729,0.001551,0.005276,0.025225,0.009141,0.209149,0.388972,0.044758,0.347590,0.606228,0.001624,0.258638,7,0.062677,0.414869,33.248181,0.048316,33.176,33.322,0.146,0.002663,39.630241,12.626917,2.457064,3.110095,41.140055,19.991351,61.877399,3.008428,2377.814453,201.671875,2578.465820,0.125,0.125,0.125,1,1
2,S10,28,178,76,24.0,1,0,0,0,0,2,1,92.743837,30.566826,723.828125,256.147262,59,0.746835,389.281709,0.001605,0.005238,0.025854,0.009290,0.202613,0.389237,0.044763,0.347590,0.606228,0.001624,0.258638,7,0.062677,0.414794,33.248677,0.048440,33.176,33.322,0.146,0.002673,39.513574,12.586034,2.439437,3.092266,41.265267,19.984413,61.873905,3.010429,2370.814453,200.614258,2585.978516,0.125,0.125,0.125,1,1
3,S10,28,178,76,24.0,1,0,0,0,0,3,1,92.652350,30.388575,723.572531,254.571460,59,0.737500,386.844992,0.001604,0.005119,0.025099,0.009718,0.203935,0.389497,0.044764,0.347590,0.606228,0.001625,0.258638,7,0.062677,0.414637,33.249171,0.048570,33.176,33.322,0.146,0.002684,39.397217,12.544515,2.423535,3.075650,41.390365,19.976378,61.870738,3.011397,2363.833008,199.660156,2593.484375,0.125,0.125,0.125,1,1
4,S10,28,178,76,24.0,1,0,0,0,0,4,1,92.629447,30.394781,723.765432,254.561813,59,0.737500,386.856825,0.001603,0.005119,0.025099,0.009718,0.203940,0.389762,0.044763,0.347590,0.606228,0.001625,0.258638,7,0.062677,0.414685,33.249662,0.048708,33.176,33.322,0.146,0.002695,39.281429,12.502663,2.409635,3.060331,41.516048,19.967527,61.868780,3.009604,2356.885742,198.826172,2601.025391,0.125,0.125,0.125,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
179812,S9,26,181,75,22.9,1,0,0,0,1,11955,4,77.176997,19.034530,818.359375,178.236851,38,0.535211,249.628355,0.000875,0.004107,0.005070,0.003737,0.810106,0.394167,0.009072,0.384254,0.434087,0.000350,0.049833,11,0.006288,0.054151,31.640050,0.011828,31.618,31.672,0.054,0.000510,25.140853,14.972495,8.471794,19.905776,47.361214,17.529328,62.172694,3.027760,1533.642578,922.116211,2841.672852,0.125,0.125,0.125,1,1
179813,S9,26,181,75,22.9,1,0,0,0,1,11956,4,77.469578,19.025559,815.321181,178.576851,39,0.549296,250.618184,0.000893,0.004579,0.005438,0.003587,0.842093,0.394463,0.010093,0.384254,0.462867,0.000377,0.078613,11,0.006288,0.062596,31.640217,0.012022,31.618,31.674,0.056,0.000522,25.268115,15.032660,8.554508,19.947584,47.234782,17.568917,62.178357,3.031119,1541.278320,927.079102,2834.086914,0.125,0.125,0.125,1,1
179814,S9,26,181,75,22.9,1,0,0,0,1,11957,4,77.527298,19.061482,814.887153,178.924628,39,0.549296,250.974663,0.000891,0.004579,0.005438,0.003587,0.842027,0.394805,0.011304,0.384254,0.473556,0.000409,0.089302,11,0.006288,0.076035,31.640392,0.012234,31.618,31.676,0.058,0.000534,25.353320,15.072298,8.665120,20.013907,

In [13]:
# second version with final_df merged with signals
final_df.rename(columns={"subject_id": "subject"}, inplace=True)
final_merged_df = pd.merge(final_df, signals, on="subject", how="left")
print(final_df.shape, "+", signals.shape, "- the double counting of subject_id")
print(final_merged_df.shape)
display(final_merged_df)

# save final_merged_df to csv
final_merged_df.to_csv("merged_survey_metadata_wrist_features.csv", index=False)

(15, 91) + (179817, 46) - the double counting of subject_id
(179817, 136)


,subject,Base_0,Base_1,Base_2,Base_3,Base_4,Base_5,Base_6,Base_7,Base_8,Base_9,Base_10,Base_11,Base_12,Base_13,Base_14,Base_15,TSST_0,TSST_1,TSST_2,TSST_3,TSST_4,TSST_5,TSST_6,TSST_7,TSST_8,TSST_9,TSST_10,TSST_11,TSST_12,TSST_13,TSST_14,TSST_15,Fun_0,Fun_1,Fun_2,Fun_3,Fun_4,Fun_5,Fun_6,Fun_7,Fun_8,Fun_9,Fun_10,Fun_11,Fun_12,Fun_13,Fun_14,Fun_15,Medi 1_0,Medi 1_1,Medi 1_2,Medi 1_3,Medi 1_4,Medi 1_5,Medi 1_6,Medi 1_7,Medi 1_8,Medi 1_9,Medi 1_10,Medi 1_11,Medi 1_12,Medi 1_13,Medi 1_14,Medi 1_15,Medi 2_0,Medi 2_1,Medi 2_2,Medi 2_3,Medi 2_4,Medi 2_5,Medi 2_6,Medi 2_7,Medi 2_8,Medi 2_9,Medi 2_10,Medi 2_11,Medi 2_12,Medi 2_13,Medi 2_14,Medi 2_15,fun_first,age,height,weight,bmi,gender_male,coffee_today_YES,sports_today_YES,smoker_YES,ill_today_YES,window_index,label,bvp_hr_mean,bvp_hr_std,bvp_hrv_mean,bvp_hrv_std,bvp_hrv_nn50,bvp_hrv_pnn50,bvp_hrv_rmssd,bvp_hrv_ulf,bvp_hrv_lf,bvp_hrv_hf,bvp_hrv_uhf,bvp_hrv_lf_hf,eda_mean,eda_std,eda_min,eda_max,eda_slope,eda_range,eda_scr_peaks,eda_scr_mean_amp,eda_scr_auc,temp_mean,temp_std,temp_min,temp_max,temp_range,temp_slope,acc_x_mean,acc_x_std,acc_y_mean,acc_y_std,acc_z_mean,acc_z_std,acc_mag_mean,acc_mag_std,acc_x_absint,acc_y_absint,acc_z_absint,acc_x_peakfreq,acc_y_peakfreq,acc_z_peakfreq,eda_scr_has_peaks,bvp_hrv_freq_valid
0,S10,2,2,1,4,2,3,3,4,1,3,6,8,6,3,6,3,1,4,2,1,1,3,3,3,1,3,0,0,0,0,0,3,1,1,1,4,2,2,1,1,4,2,2,2,1,8,2,2,3,3,2,3,1,1,1,1,2,1,0,0,0,0,0,2,1,1,1,3,1,1,1,1,2,1,0,0,0,0,0,3,1,28,178,76,24.0,1,0,0,0,0,0,1,92.222633,30.733178,729.552469,259.659106,60,0.750000,391.919729,0.001551,0.005276,0.025225,0.009141,0.209149,0.388696,0.044754,0.347590,0.606228,0.001623,0.258638,7,0.062677,0.415006,33.247683,0.048199,33.176,33.322,0.146,0.002654,39.746387,12.666104,2.475993,3.129106,41.014355,19.997278,61.879907,3.012126,2384.783203,202.807617,2570.923828,0.125,0.125,0.125,1,1
1,S10,2,2,1,4,2,3,3,4,1,3,6,8,6,3,6,3,1,4,2,1,1,3,3,3,1,3,0,0,0,0,0,3,1,1,1,4,2,2,1,1,4,2,2,2,1,8,2,2,3,3,2,3,1,1,1,1,2,1,0,0,0,0,0,2,1,1,1,3,1,1,1,1,2,1,0,0,0,0,0,3,1,28,178,76,24.0,1,0,0,0,0,1,1,92.222633,30.733178,729.552469,259.659106,60,0.750000,391.919729,0.001551,0.005276,0.025225,0.009141,0.209149,0.388972,0.044758,0.347590,0.606228,0.001624,0.258638,7,0.062677,0.414869,33.248181,0.048316,33.176,33.322,0.146,0.002663,39.630241,12.626917,2.457064,3.110095,41.140055,19.991351,61.877399,3.008428,2377.814453,201.671875,2578.465820,0.125,0.125,0.125,1,1
2,S10,2,2,1,4,2,3,3,4,1,3,6,8,6,3,6,3,1,4,2,1,1,3,3,3,1,3,0,0,0,0,0,3,1,1,1,4,2,2,1,1,4,2,2,2,1,8,2,2,3,3,2,3,1,1,1,1,2,1,0,0,0,0,0,2,1,1,1,3,1,1,1,1,2,1,0,0,0,0,0,3,1,28,178,76,24.0,1,0,0,0,0,2,1,92.743837,30.566826,723.828125,256.147262,59,0.746835,389.281709,0.001605,0.005238,0.025854,0.009290,0.202613,0.389237,0.044763,0.347590,0.606228,0.001624,0.258638,7,0.062677,0.414794,33.248677,0.048440,33.176,33.322,0.146,0.002673,39.513574,12.586034,2.439437,3.092266,41.265267,19.984413,61.873905,3.010429,2370.814453,200.614258,2585.978516,0.125,0.125,0.125,1,1
3,S10,2,2,1,4,2,3,3,4,1,3,6,8,6,3,6,3,1,4,2,1,1,3,3,3,1,3,0,0,0,0,0,3,1,1,1,4,2,2,1,1,4,2,2,2,1,8,2,2,3,3,2,3,1,1,1,1,2,1,0,0,0,0,0,2,1,1,1,3,1,1,1,1,2,1,0,0,0,0,0,3,1,28,178,76,24.0,1,0,0,0,0,3,1,92.652350,30.388575,723.572531,254.571460,59,0.737500,386.844992,0.001604,0.005119,0.025099,0.009718,0.203935,0.389497,0.044764,0.347590,0.606228,0.001625,0.258638,7,0.062677,0.414637,33.249171,0.048570,33.176,33.322,0.146,0.002684,39.397217,12.544515,2.423535,3.075650,41.390365,19.976378,61.870738,3.011397,2363.833008,199.660156,2593.484375,0.125,0.125,0.125,1,1
4,S10,2,2,1,4,2,3,3,4,1,3,6,8,6,3,6,3,1,4,2,1,1,3,3,3,1,3,0,0,0,0,0,3,1,1,1,4,2,2,1,1,4,2,2,2,1,8,2,2,3,3,2,3,1,1,1,1,2,1,0,0,0,0,0,2,1,1,1,3,1,1,1,1,2,1,0,0,0,0,0,3,1,28,178,76,24.0,1,0,0,0,0,4,1,92.629447,30.394781,723.765432,254.561813,59,0.737500,386.856825,0.001603,0.005119,0.025099,0.009718,0.203940,0.389762,0.044763,0.347590,0.606228,0.001625,0.258638,7,0.062677,0.414685,33.249662,0.048708,33.176,33.322,0.146,0.002695,39.281429,12.502663,2.409635,3.060331